# 🦆 KOSTRA AI Workbench — DuckDB Utforskningsnotatblokk
Dette notatet lar deg koble direkte til **`kostra_prod.duckdb`** for ad-hoc analyse, avviksutskrift og kildekontroll uten å forstyrre produksjonspipelinen.

### 1. Importer biblioteker og koblet til DuckDB (Skrivebeskyttet modus)
Ved å sette `read_only=True` kan du kjøre analyser i Jupyter samtidig som Streamlit eller ETL-prosesser oppdaterer databasen.

In [6]:
conn.close()
print("🔒 Tilkobling til DuckDB lukket.")

🔒 Tilkobling til DuckDB lukket.


.\duckdb -readonly data/database/kostra_prod.duckdb,  .\duckdb data/database/kostra_prod.duckdb

.maxrows 0
.maxwidth 0
.mode box

.tables – Lister opp alle tabeller og views.
.schema – Viser CREATE TABLE/VIEW-DDL for hele databasen.
.schema stg_fact_komskole – Viser skjemastrukturen til én spesifikk tabell.
.mode box – Endrer utskriftsformatet til pene tabellrammer i terminalen.
.mode markdown – Formaterer spørringsresultatet direkte som Markdown.
.timer on – Viser eksakt kjøretid (i millisekunder) for hver SQL-spørring.
.exit (eller .quit) – Avslutter DuckDB CLI.

Kjør avviksspørring og vis direkte i terminalen:
duckdb -readonly data/database/kostra_prod.duckdb "SELECT * FROM vw_avvik_summary LIMIT 5;"
Profilere en tabell (min/maks/null-verdier):
duckdb -readonly data/database/kostra_prod.duckdb "SUMMARIZE stg_fact_komskole;"
Eksportere et resultat direkte til Parquet:
duckdb -readonly data/database/kostra_prod.duckdb "COPY (SELECT * FROM vw_avvik_summary) TO 'data/export/avvik.parquet' (FORMAT PARQUET);"
Eksportere et resultat til CSV:
duckdb -readonly data/database/kostra_prod.duckdb "COPY (SELECT * FROM vw_avvik_summary) TO 'data/export/avvik.csv' (HEADER, DELIMITER ',');"

In [ ]:
import pandas as pd

# Fjern begrensninger på kolonner, rader og tekstlengde
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
#Hvis du vil skrive ut alt til konsollen som ren tekst uten avkutting:
print(df.to_string())

In [5]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# Sett sti til databasen i Kom - G2 prosjektet
DB_PATH = "./data/database/kostra_prod.duckdb"

# Koble til DuckDB i skrivebeskyttet modus
conn = duckdb.connect(DB_PATH, read_only=True)
print("✅ Koblet til DuckDB i read-only modus!")

✅ Koblet til DuckDB i read-only modus!


### 2. Lister alle tabeller og views i databasen

In [2]:

df_ml = conn.execute("SELECT * FROM modelready_fact").fetchdf()

In [3]:
df_ml

,accounting_period,year,month,quarter,is_summer_season,is_year_end,funksjon_code,funksjon_navn,konto_code,kontonavn,...,rolling_3m_avg_regnskap_nok,rolling_3m_std_regnskap_nok,target_next_month_regnskap,burn_rate_pct,budget_coverage_ratio,anomaly_z_score_3m,abs_monthly_avvik_nok,log_monthly_regnskap_nok,target_status_code,target_is_overbudget
0,2024-01-01,2024,1,1,0,0,2020,Grunnskoleopplæring,104000,Overtidslønn,...,2250.000000,0.000000,0.0,1.0,1.0,0.000000,2250.0,7.718685,0,0
1,2024-02-01,2024,2,1,0,0,2020,Grunnskoleopplæring,104000,Overtidslønn,...,1125.000000,1590.990258,0.0,1.0,1.0,-0.707107,0.0,0.000000,0,0
2,2024-03-01,2024,3,1,0,0,2020,Grunnskoleopplæring,104000,Overtidslønn,...,750.000000,1299.038106,0.0,1.0,1.0,-0.577350,0.0,0.000000,0,0
3,2024-04-01,2024,4,2,0,0,2020,Grunnskoleopplæring,104000,Overtidslønn,...,0.000000,0.000000,0.0,1.0,1.0,0.000000,0.0,0.000000,0,0
4,2024-05-01,2024,5,2,1,0,2020,Grunnskoleopplæring,104000,Overtidslønn,...,0.000000,0.000000,0.0,1.0,1.0,0.000000,0.0,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2251,2024-02-01,2024,2,1,0,0,2540,Helse- og omsorgstjenester i institusjon,124010,Drifts- / serviceavtaler - kontormaskiner,...,572.600000,0.000000,1103.8,1.0,1.0,0.000000,572.6,6.350187,0,0
2252,2024-04-01,2024,4,2,0,0,2540,Helse- og omsorgstjenester i institusjon,124010,Drifts- / serviceavtaler - kontormaskiner,...,838.200000,375.615122,-984.4,1.0,1.0,0.707107,1103.8,7.006514,0,0
2253,2024-05-01,2024,5,2,1,0,2540,Helse- og omsorgstjenester i institusjon,124010,Drifts- / serviceavtaler - kontormaskiner,...,230.666667,1085.280431,-984.4,1.0,1.0,-1.119588,984.4,6.892032,1,1
2254,2024-04-01,2024,4,2,0,0,2540,Helse- og omsorgstjenester i institusjon,124023,Driftsavtaler - medisinsk utstyr,...,0.000000,0.000000,0.0,1.0,1.0,0.000000,0.0,0.000000,0,0


In [17]:
tables_df = conn.execute("SHOW TABLES;").fetchdf()
print(tables_df)

                   name
0             audit_log
1            dim_ansvar
2              dim_date
3   dim_kostra_funksjon
4  dim_kostra_kontoplan
5         dim_sikkerhet
6              stg_fact
7     stg_fact_komskole
8      vw_avvik_summary


### 3. Utforsk topp 10 kritiske avvik (`vw_avvik_summary`)

In [7]:

def get_duckdb_metadata(
    db_path: str = "data/database/kostra_prod.duckdb",
) -> pd.DataFrame:
  """Ekstraherer komplett metadata for alle tabeller og views i DuckDB."""
  conn = duckdb.connect(db_path, read_only=True)

  # 1. Hent alle tabeller og views
  tables_df = conn.execute("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
        ORDER BY table_type, table_name;
    """).fetchdf()

  metadata_list = []

  for _, row in tables_df.iterrows():
    schema = row["table_schema"]
    tbl_name = row["table_name"]
    tbl_type = row["table_type"]

    # Hent radantall per tabell/view
    try:
      row_count = conn.execute(
          f'SELECT COUNT(*) FROM "{schema}"."{tbl_name}"'
      ).fetchone()[0]
    except Exception:
      row_count = None

    # Hent kolonnemetadata
    cols_df = conn.execute(f"""
            SELECT 
                column_name, 
                data_type, 
                is_nullable,
                ordinal_position
            FROM information_schema.columns
            WHERE table_schema = '{schema}' AND table_name = '{tbl_name}'
            ORDER BY ordinal_position;
        """).fetchdf()

    for _, col in cols_df.iterrows():
      metadata_list.append({
          "schema": schema,
          "table_name": tbl_name,
          "table_type": tbl_type,
          "row_count": row_count,
          "column_position": col["ordinal_position"],
          "column_name": col["column_name"],
          "data_type": col["data_type"],
          "is_nullable": col["is_nullable"],
      })

  conn.close()
  return pd.DataFrame(metadata_list)


# Kjøring og visning av resultater
if __name__ == "__main__":
  df_meta = get_duckdb_metadata("data/database/kostra_prod.duckdb")

  # Vis oppsummering per tabell
  print("=== TABELLOVERSIKT ===")
  summary = df_meta.groupby(
      ["table_name", "table_type", "row_count"]
  ).size().reset_index(name="antall_kolonner")
  print(summary.to_string(index=False))

  # Eksporter metadata til Excel/CSV om ønskelig
  # df_meta.to_excel("data/export/database_metadata.xlsx", index=False)

=== TABELLOVERSIKT ===
          table_name table_type  row_count  antall_kolonner
           audit_log BASE TABLE          0               10
          dim_ansvar BASE TABLE         39                8
            dim_date BASE TABLE        366                8
 dim_kostra_funksjon BASE TABLE         61               13
dim_kostra_kontoplan BASE TABLE        172               18
       dim_sikkerhet BASE TABLE          4                3
   stg_fact_komskole BASE TABLE       2256                9
    vw_avvik_summary       VIEW        106                8


In [12]:
query = """
SELECT 
    table_name,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'main'
ORDER BY table_name, ordinal_position;
"""

df_metadata = conn.execute(query).fetchdf()
print(df_metadata)

          table_name       column_name      data_type is_nullable
0          audit_log            log_id        VARCHAR          NO
1          audit_log         timestamp      TIMESTAMP         YES
2          audit_log           user_id        VARCHAR         YES
3          audit_log        query_text        VARCHAR         YES
4          audit_log   sektor_funksjon        VARCHAR         YES
..               ...               ...            ...         ...
72  vw_avvik_summary         Kontonavn        VARCHAR         YES
73  vw_avvik_summary      Sum_Budsjett  DECIMAL(38,1)         YES
74  vw_avvik_summary      Sum_Regnskap         DOUBLE         YES
75  vw_avvik_summary          Avvik_Kr         DOUBLE         YES
76  vw_avvik_summary  Status_Indikator        VARCHAR         YES

[77 rows x 4 columns]


In [ ]:
avvik_query = """
SELECT 
    Funksjon_Kode,
    KOSTRA_Funksjon_Navn,
    Kontokode_6Siffer,
    Kontonavn,
    Sum_Budsjett,
    Sum_Regnskap,
    Avvik_Kr,
    Status_Indikator
FROM vw_avvik_summary
ORDER BY Avvik_Kr ASC
LIMIT 10;
"""
df_top_avvik = conn.execute(avvik_query).fetchdf()
df_top_avvik

### 4. Visualiser avvik for topp 10 kontooppføringer

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(df_top_avvik['Kontonavn'] + ' (' + df_top_avvik['Funksjon_Kode'] + ')', df_top_avvik['Avvik_Kr'] / 1e6, color='crimson')
plt.xlabel('Avvik (Millioner NOK)')
plt.ylabel('Konto & KOSTRA Funksjon')
plt.title('Topp 10 største merforbruk / negativt avvik')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 5. Sjekk fordeling av statusindikatorer (🔴 / 🟡 / 🟢)

In [ ]:
status_query = """
SELECT 
    Status_Indikator,
    COUNT(*) AS Antall_Oppføringer,
    SUM(Avvik_Kr) AS Totalt_Avvik_NOK
FROM vw_avvik_summary
GROUP BY Status_Indikator
ORDER BY Totalt_Avvik_NOK ASC;
"""
conn.execute(status_query).fetchdf()

### 6. Analyser rådata per kilde-fil (`source_file`)
DuckDB bevarer alle bevarte dimensjoner og filopphav i `stg_fact_komskole`.

In [ ]:
source_summary = """
SELECT 
    source_file,
    COUNT(*) AS Antall_Rader,
    SUM(Budsjett) AS Totalt_Budsjett,
    SUM(Regnskap) AS Totalt_Regnskap
FROM stg_fact_komskole
GROUP BY source_file
ORDER BY source_file ASC;
"""
conn.execute(source_summary).fetchdf()

### 7. Lukk tilkoblingen

In [4]:
conn.close()
print("🔒 Tilkobling til DuckDB lukket.")

🔒 Tilkobling til DuckDB lukket.
